In [0]:
import mlflow
from mlflow import MlflowClient
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "fraud_lakehouse")
CATALOG = dbutils.widgets.get("catalog")
MODEL_NAME = f"{CATALOG}.ml.fraud_detector"
TARGET = f"{CATALOG}.ml.fraud_scores"
THRESHOLD = 0.5
FEATURES = ["amount", "distance_from_home_km", "hour_of_day", "is_night", "is_online",
            "is_high_risk_category", "txn_count_1h", "amount_sum_1h", "amount_vs_30d_avg"]

mlflow.set_registry_uri("databricks-uc")
version = MlflowClient().get_model_version_by_alias(MODEL_NAME, "champion").version
model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")

features = spark.table(f"{CATALOG}.gold.fraud_features")
day = features.agg(F.max("date_key")).first()[0]
pdf = features.where(F.col("date_key") == day).select("transaction_id", "date_key", *FEATURES).toPandas()

pdf["fraud_score"] = model.predict_proba(pdf[FEATURES])[:, 1]
pdf["is_flagged"] = pdf["fraud_score"] >= THRESHOLD
pdf["model_version"] = str(version)

scores = (
    spark.createDataFrame(pdf[["transaction_id", "date_key", "fraud_score", "is_flagged", "model_version"]])
    .withColumn("scored_at", F.current_timestamp())
)

# rerunning the same day replaces that day's scores instead of duplicating them
writer = scores.write.mode("overwrite")
if spark.catalog.tableExists(TARGET):
    writer = writer.option("replaceWhere", f"date_key = {day}")
writer.saveAsTable(TARGET)

print(f"scored {len(pdf):,} transactions for {day} with v{version}, flagged {int(pdf['is_flagged'].sum())}")